In [5]:
import os
import json
from collections import Counter
from pathlib import Path
import numpy as np
import torch
from datasets import Dataset
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModel
from vul_detector import VulDetector
from vul_trainer import VulTrainerManual
import time
from torch.utils.data import WeightedRandomSampler

In [18]:

start = time.time()
time.sleep(5)
end = time.time()
print(f"Elapsed time: {end - start} seconds")

Elapsed time: 5.0053770542144775 seconds


In [6]:
# -------------------------
# 0) Load JSONL -> HF Dataset
# -------------------------
def load_jsonl(path):
    data = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                data.append(json.loads(line))
    return data

# Set this to your folder containing train/val/test jsonl from preprocessing
DATASET_DIR = os.path.join(os.getcwd(), "data")
assert os.path.isdir(DATASET_DIR), f"Dataset dir not found: {DATASET_DIR}"

train_dataset = Dataset.from_list(load_jsonl(os.path.join(DATASET_DIR, "train.jsonl")))
valid_dataset = Dataset.from_list(load_jsonl(os.path.join(DATASET_DIR, "val.jsonl")))
test_dataset = Dataset.from_list(load_jsonl(os.path.join(DATASET_DIR, "test.jsonl")))

print("Loaded:", len(test_dataset))
print("Example row keys:", test_dataset.column_names)
print("Example:", {k: test_dataset[0][k] for k in ["id", "project", "target", "answer_text"]})

Loaded: 22817
Example row keys: ['id', 'project', 'target', 'func_clean', 'prompt', 'answer_text']
Example: {'id': 'c6521fc0944b068e', 'project': 'shibboleth', 'target': 1, 'answer_text': 'vulnerable'}


In [20]:
sum(1 for x in test_dataset['target'] if x == 1)

846

In [21]:

sum(1 for x in test_dataset['target'] if x == 0)

21971

In [7]:
# -------------------------
# 2) Tokenizer + Map
# -------------------------
model_name = "microsoft/unixcoder-base"
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)

MAX_LEN = 1024  # start 256; try 384 if needed (1024 will likely OOM)

# def tokenize_batch(batch):
#     texts = [c.strip() for c in batch["func_clean"]]

#     enc = tokenizer(
#         texts,
#         truncation=True,
#         max_length=MAX_LEN,
#         padding="max_length",
#         add_special_tokens=True,
#     )

#     # Labels: prefer numeric target
#     if "target" in batch:
#         enc["labels"] = [int(x) for x in batch["target"]]
#     else:
#         enc["labels"] = [
#             1 if a.strip().lower() == "vulnerable" else 0
#             for a in batch["answer_text"]
#         ]
#     return enc
def tokenize_batch(batch):
    texts = [c.strip() for c in batch["func_clean"]]
    enc = tokenizer(texts, truncation=True, max_length=MAX_LEN, padding="max_length")

    # Labels: prefer numeric target
    enc["labels"] = [int(x) for x in batch["target"]]
    return enc


train_tok = train_dataset.map(
    tokenize_batch,
    batched=True,
    remove_columns=train_dataset.column_names,
)
valid_tok = valid_dataset.map(
    tokenize_batch,
    batched=True,
    remove_columns=valid_dataset.column_names,
)
test_tok = test_dataset.map(
    tokenize_batch,
    batched=True,
    remove_columns=test_dataset.column_names,
)

Map: 100%|██████████| 22817/22817 [00:04<00:00, 4899.72 examples/s]


In [10]:
train_tok['input_ids']

Column([[0, 2456, 23589, 126, 6013, 426, 201, 127, 317, 394, 399, 317, 394, 1534, 22067, 136, 385, 431, 408, 201, 137, 408, 1951, 132, 894, 181, 6874, 408, 12351, 136, 145, 317, 969, 462, 400, 7978, 135, 181, 459, 181, 1724, 126, 201, 127, 1451, 16451, 135, 181, 136, 181, 4482, 698, 317, 411, 22067, 136, 550, 400, 6013, 181, 5934, 5835, 2044, 181, 5122, 181, 4317, 210, 7978, 135, 181, 1419, 156, 509, 317, 466, 483, 9383, 181, 5934, 5835, 2044, 181, 5122, 181, 5835, 3528, 649, 16451, 135, 181, 1419, 156, 181, 5835, 3528, 145, 317, 394, 483, 22067, 136, 145, 317, 316, 211, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,

In [ ]:

# # compute embeddings for train set, apply SMOTE to embeddings, produce a balanced DataLoader

# def balance_embeddings_with_smote(batcher, emb_model, embed_device):
#     emb_model.to(embed_device)
#     emb_model.eval()
#     embs = []
#     ys = []
#     with torch.no_grad():
#         for b in batcher:
#             input_ids = b["input_ids"].to(embed_device)
#             attention_mask = b["attention_mask"].to(embed_device)
#             out = emb_model(input_ids=input_ids, attention_mask=attention_mask, return_dict=True)
#             # use pooler_output if available else mean-pool last hidden state
#             if hasattr(out, "pooler_output") and out.pooler_output is not None:
#                 pooled = out.pooler_output
#             else:
#                 pooled = out.last_hidden_state.mean(dim=1)
#             embs.append(pooled.cpu())
#             ys.append(b["labels"].cpu())

#     X = torch.cat(embs).numpy()
#     y = torch.cat(ys).numpy()

#     print("Before SMOTE:", Counter(y))

#     smote = SMOTE(sampling_strategy="auto", k_neighbors=3, random_state=42)
#     X_smote, y_smote = smote.fit_resample(X, y)

#     print("After SMOTE:", Counter(y_smote))

#     # create a DataLoader of embeddings+labels (for training a classifier on embeddings)
#     X_smote_t = torch.tensor(X_smote, dtype=torch.float32)
#     y_smote_t = torch.tensor(y_smote, dtype=torch.long)
#     smote_dataset = torch.utils.data.TensorDataset(X_smote_t, y_smote_t)
#     smote_loader = DataLoader(smote_dataset, batch_size=32, shuffle=True)
#     return smote_loader


In [11]:
train_dataset[0]["target"]

1

In [12]:
# -------------------------
# 3) DataLoaders + class weights
# -------------------------
# embed_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# print("Embedding device:", embed_device)
# emb_model = AutoModel.from_pretrained(model_name)

columns = ["input_ids", "attention_mask", "labels"]
train_tok.set_format(type="torch", columns=columns)
valid_tok.set_format(type="torch", columns=columns)
test_tok.set_format(type="torch", columns=columns)

labels = np.array(train_dataset["target"]).astype(np.int64)

label_counts = Counter(labels)
total = sum(label_counts.values())

class_weights = np.array([total / (2 * label_counts[i]) for i in range(2)], dtype=np.float64)
sample_weights = class_weights[labels]

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

train_loader = DataLoader(train_tok, batch_size=16, sampler=sampler)
val_loader = DataLoader(valid_tok, batch_size=32)
test_loader = DataLoader(test_tok, batch_size=32)


print("Class weights:", class_weights)
print("Batches -> train:", len(train_loader), "val:", len(val_loader), "test:", len(test_loader))

Class weights: [ 0.51396701 18.39932603]
Batches -> train: 5460 val: 2045 test: 714


In [30]:
idx = list(iter(WeightedRandomSampler(sample_weights, num_samples=5000, replacement=True)))
drawn = labels[idx]
print(Counter(drawn))

Counter({np.int64(0): 2590, np.int64(1): 2410})


In [13]:
for batch in train_loader:
    inputs = batch["input_ids"]
    labels = batch["labels"]
    print(inputs)
    print(labels)
    print("Sample batch shapes:", inputs.shape, labels.shape)
    
    # Count positive and negative labels
    num_positive = (labels == 1).sum().item()
    num_negative = (labels == 0).sum().item()
    print(f"Positive (vulnerable): {num_positive}")
    print(f"Negative (non-vulnerable): {num_negative}")
    break

tensor([[    0,   932,   554,  ...,     1,     1,     1],
        [    0,   430, 23589,  ...,  3736,   581,     2],
        [    0,  1683, 23589,  ...,     1,     1,     1],
        ...,
        [    0,   430, 23589,  ...,   181,  2757,     2],
        [    0,  8321,   126,  ...,  6840,  5692,     2],
        [    0,   895,  9431,  ...,     1,     1,     1]])
tensor([0, 1, 0, 1, 1, 0, 1, 1, 1, 1, 0, 0, 1, 1, 1, 0])
Sample batch shapes: torch.Size([16, 1024]) torch.Size([16])
Positive (vulnerable): 10
Negative (non-vulnerable): 6


In [ ]:
# -------------------------
# 4) Train vulnerability detector
# -------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

model = VulDetector(
    model_name=model_name,
    num_labels=2,
    pooling="mean",
    head_type="mlp",
    hidden_dropout=0.1
)
#model = VulDetector(model_name=model_name, num_labels=2)
trainer = VulTrainerManual(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    class_weights=class_weights,
    learning_rate=1e-5,
    num_epochs=1,
    loss_type="cross_entropy",
    focal_gamma=1.5,
)

trainer.train()

In [ ]:
# -------------------------
# 5) Evaluate on validation/test splits
# -------------------------
def evaluate_loader(loader, model, device, criterion, threshold=0.5):
    model.eval()
    total_loss = 0.0
    all_probs, all_preds, all_labels = [], [], []

    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            labels = batch.pop("labels")

            outputs = model(**batch)
            logits = outputs.logits
            loss = criterion(logits, labels)
            total_loss += loss.item()
    
            probs = torch.softmax(logits, dim=1)[:, 1]        # P(vulnerable)
            preds = (probs >= threshold).long()               # threshold decision

            all_probs.extend(probs.cpu().tolist())
            all_preds.extend(preds.cpu().tolist())
            all_labels.extend(labels.cpu().tolist())

    metrics = trainer.compute_metrics(np.array(all_preds), np.array(all_labels))
    metrics["loss"] = total_loss / max(len(loader), 1)
    return metrics, np.array(all_probs), np.array(all_labels)

best_ckpts = sorted(Path(".").glob("best_model_epoch_*.pt"), key=lambda p: p.stat().st_mtime)
if best_ckpts:
    best_ckpt = best_ckpts[-1]
    model.load_state_dict(torch.load(best_ckpt, map_location=device))
    model.to(device)
    print(f"Loaded best checkpoint: {best_ckpt}")
else:
    print("No saved checkpoints found; evaluating current model state.")

val_metrics = evaluate_loader(val_loader, model, device, trainer.criterion)[0]
print("Validation metrics:", val_metrics)

test_metrics = evaluate_loader(test_loader, model, device, trainer.criterion)[0]
print("Test metrics:", test_metrics)